# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates data exploration and processing of the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @ids and fields

record_sets = getattr(metadata, 'record_set', [])
if not record_sets:
    print("No record sets found.\nTrying to inspect via the underlying Croissant JSON...")
    # Optionally, parse the JSON if available. In many real datasets, use metadata.to_json()['recordSet']
    # Let's fetch record sets from metadata itself (depending on croissant version)
    meta_json = metadata.to_json() if hasattr(metadata, 'to_json') else {}
    record_sets = meta_json.get('recordSet', [])
if not record_sets:
    raise Exception('No record sets found in dataset.')

print(f"Found {len(record_sets)} record set(s):\n")
if isinstance(record_sets[0], dict):
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    record_set_ids = [rs for rs in record_sets]
for rs in record_set_ids:
    print(f"- Record set @id: {rs}")

In [ ]:
# Show all field @ids in each record set

for rs in record_set_ids:
    print(f"\nRecord set: {rs}")
    fields = dataset.fields(record_set=rs)
    for field in fields:
        print(f"  - field @id: {field['@id']} | name: {field.get('name', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Uses record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into pandas DataFrames

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found in this record set.")

In [ ]:
# Display columns and preview main DataFrame

# For this dataset, main survey data is typically in the first record set
main_rs = record_set_ids[0]
df = dataframes[main_rs]
print(f"Columns in main record set ({main_rs}):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing numeric fields, grouping, and removing outliers.

In [ ]:
# Pick a numeric field for numeric analysis, e.g. total_GAD_7_score or total_PHQ_9_score
# These columns (names) and their ids can be checked by looking at the fields above.

numeric_field_id = 'total_GAD_7_score'  # Adjust to actual @id from above if needed
if numeric_field_id not in df.columns:
    print(f"Field {numeric_field_id} not found in main DataFrame. Falling back to first numeric column.")
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns found: {numeric_cols}")
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
    else:
        raise ValueError("No numeric columns for demonstration.")
print(f"Using numeric field: {numeric_field_id}")

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical/grouping field (e.g. gender or village)
group_field_id = 'gender'  # Replace with actual @id for gender if needed
if group_field_id not in df.columns:
    cat_fields = df.select_dtypes(include=['object']).columns
    print(f"Fallback: Categorical columns: {cat_fields.tolist()}")
    group_field_id = cat_fields[0]

print(f"\nGrouping by: {group_field_id}")
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df)
else:
    print(f"Field {group_field_id} not in DataFrame columns.")

## 5. Visualization
Visualize data distributions and relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Example: Boxplot of numeric field grouped by categorical variable
if group_field_id in df.columns:
    plt.figure(figsize=(9, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the Kilifi County FAIR² Mental Health Survey dataset using `mlcroissant`. We performed initial inspection, field analysis, filtering, normalization, group-based statistics, and visualizations. This process can be extended to conduct in-depth analysis of mental health survey data for research and policy applications.